In [1]:
# Import librerie e setup percorsi
from datetime import datetime
import glob
import json
import os
import sys
import time
import uuid

from langchain_community.document_loaders import PyPDFLoader
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_ollama import ChatOllama

# Import del modulo custom src/
PROJECT_ROOT = os.path.expanduser("~/tesi_graphrag")
sys.path.append(PROJECT_ROOT)
from src.config import (
    EMBEDDING_MODEL,
    QDRANT_URL,
    LLM_MODEL,
    OLLAMA_URL,
)

# Configurazione ingestione e salvataggio registro
REGISTRY_PATH = os.path.join(PROJECT_ROOT, "data/collection_registry.json")
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

/tmp/ipykernel_476459/4210865576.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
# Percorsi e configurazioni -- DataSet 1 (ds1) -- Cambiare se necessario
DATASET_DIR = os.path.join(PROJECT_ROOT, "data/raw/ds1")
COLLECTION_NAME = "ds1_multilingual_base"

print(f"Dataset directory: {DATASET_DIR} | Embeggind model: {EMBEDDING_MODEL}")

Dataset directory: /home/jovyan/tesi_graphrag/data/raw/ds1 | Embeggind model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2


In [3]:
# Caricamento PDF e Chunking

# Mappa dell'anno di pubblicazione ricavata dai paper
YEAR_MAP = {
    "GraphRAG_Microsoft": 2024,
    "Self_RAG": 2023,
    "CRAG_Corrective_RAG": 2024,
    "RAPTOR_Hierarchical_RAG": 2024,
    "Original_RAG_Lewis": 2020,
    "HyDE_Hypothetical_Embeddings": 2022,
    "KG_RAG_Survey": 2024,
    "FLARE_Active_RAG": 2023,
    "Knowledge_Graph_Prompting": 2023,
    "RAG_vs_FineTuning": 2024,
}

#! trova e elenca tutti i file contenuti nel percorso specificato
pdf_files = glob.glob(os.path.join(DATASET_DIR, "*.pdf"))
print(f"Trovati {len(pdf_files)} PDF in {DATASET_DIR}")

if not pdf_files:
    sys.exit(f"Nessun PDF trovato in {DATASET_DIR}. Controlla il percorso prima di procedere.")

# Configurazione Text Splitter (256 token / 1000ca. caratteri, overlap di 150 - misurato in caratteri) 
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, length_function=len
)

all_docs = []
# LLM per tag ipotizzati
llm_tagger = ChatOllama(model=LLM_MODEL, base_url=OLLAMA_URL, temperature=0)
dataset_global_tags = set()

for pdf_path in pdf_files:
    filename = os.path.basename(pdf_path).replace(".pdf", "").strip()

    try:
        loader = PyPDFLoader(pdf_path)
        pages = loader.load()
    except Exception as e:
        print(f"!!!>  Errore nel caricamento di '{filename}': {e} >>> file saltato")
        continue

    """ AUTO_ASSEGNAZIONE TAG
    # Estrazione TAG dal documento
    #  Passaggio della prima pagina pensato per documenti scientifici,
    #  in modo che possa leggere titolo, abstract e eventuale breve descrizione
    sample_text = pages[0].page_content if pages else ""
    prompt_tags = (
        f"Sei un assistente esperto nell'analisi e categorizzazione di documenti di qualsiasi dominio (scientifico, tecnico, umanistico o legale).\n"
        f"Estrai da 2 a 4 macro-tag o argomenti chiave principali per il documento '{filename}'.\n\n"
        f"Testo di riferimento (Abstract/Inizio):\n{sample_text}\n\n"
        f"REGOLE DI FORMATTAZIONE:\n"
        f"1. Per ogni tag, indica prima il concetto principale e tra parentesi il contesto o la sotto-area specifica.\n"
        f"2. Mantieni i termini tecnici nella loro lingua d'origine (es. inglese se il documento è in inglese).\n"
        f"3. ESEMPI DI FORMATO CORRETTO DA DIVERSI DOMINI:\n"
        f"   - Medicina: 'Cardiologia (Scompenso Cardiaco)', 'Farmacologia (Terapie Biologiche)'\n"
        f"   - Diritto: 'Diritto Contrattuale (Inadempimento)', 'Giurisprudenza (Responsabilità Civile)'\n"
        f"   - Informatica: 'Reti Neurali (Transformer)', 'Database (Indicizzazione B-Tree)'\n\n"
        f"Rispondi ESCLUSIVAMENTE con la lista dei tag separati da virgola, senza alcuna introduzione, numerazione o testo aggiuntivo."
    )
    try:
        doc_tags_raw = llm_tagger.invoke(prompt_tags).content.strip()
        doc_tags = [t.strip() for t in doc_tags_raw.split(",") if t.strip()]
        dataset_global_tags.update(doc_tags)
    except Exception:
        doc_tags = []

    print(f">> Assegnati a {filename}: {doc_tags} tramite {LLM_MODEL}")
    """
    
    # Split in chunk
    chunks = text_splitter.split_documents(pages)

    for idx, chunk in enumerate(chunks):
        page_num = chunk.metadata.get("page", 0)

        # Creazione dell'ID univoco strutturato
        #  Il valore di fallback è doc_chunk_n, dove n è un numero globale di ciò che lo genera.
        #  Per evitare quella che sarebbe inevitabile ambiguità, è bene che non si ricada mai nel fallback
        chunk_id = f"{filename}_chunk_{idx}"

        year = YEAR_MAP.get(filename)
        if year is None and idx == 0:
            print(f"!!!>  '{filename}' non presente in YEAR_MAP, uso default 2024")
        year = year or 2024

        # Mantiene i metadati originali e aggiunge i nuovi campi di interesse
        chunk.metadata.update(
            {
                "chunk_id": chunk_id,
                "doc_id": filename,
                "page": int(page_num),
                "year": year,
                "chunk_index": int(idx),
                "user_tags": [],
            }
        )

        all_docs.append(chunk)

print(f"Totale chunk generati: {len(all_docs)}")

Trovati 10 PDF in /home/jovyan/tesi_graphrag/data/raw/ds1
>> Assegnati a KG_RAG_Survey: ['Intelligenza Artificiale (Apprendimento Automatico)', 'Ragionamento (Ragionamento Multi-Salto)', 'Grafici di Conoscenza (Evaluazione di Chain-of-Thought).'] tramite llama3.1
>> Assegnati a CRAG_Corrective_RAG: ['Intelligenza Artificiale (Reti Neurali', 'Modelli di Linguaggio)', 'Generazione di Testi (Hallucinazioni', 'RAG)', 'Retrieval di Informazioni (Evaluatore di Qualità', 'Ricerca Web)'] tramite llama3.1
>> Assegnati a RAPTOR_Hierarchical_RAG: ['Informatica (Reti Neurali)', "Informatica (Ricerca dell'Informazione)", 'Informatica (Intelligenza Artificiale)'] tramite llama3.1


KeyboardInterrupt: 

In [4]:
# Indicizzazione vettoriale su Qdrant
client = QdrantClient(url=QDRANT_URL)
embeddings = FastEmbedEmbeddings(model_name=EMBEDDING_MODEL)

# Calcola automaticamente la dimensione corretta del modello attivo (es. 768)
vector_dim = len(embeddings.embed_query("test"))

# Crea la collezione solo se non esiste già
#  Se già esistente, gli uuid deterministici calcolati la modificheranno con l'upsert
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=vector_dim, distance=Distance.COSINE),
    )


vectorstore = QdrantVectorStore(
    client=client, collection_name=COLLECTION_NAME, embedding=embeddings
)

# Caricamento di tutti i chunk su Qdrant
#  Inserimento in batch di 25 chunk alla volta per vedere
#  se il processo si è bloccato o sta funzionando; + timer
batch_size = 25
total_docs = len(all_docs)
total_start_time = time.perf_counter()

print(
    f"\n>> Avvio vettorizzazione di {total_docs} chunk ({batch_size} chunk per batch)"
)

# Ciclo di inserimento in batch con timer parziali e progressivo
for batch_idx, i in enumerate(range(0, total_docs, batch_size), start=1):
    batch_start_time = time.perf_counter()

    batch = all_docs[i : i + batch_size]

    # ID deterministico basato su namespace UUID + chunk_id
    #  Permette di riaggiornare la collection tramite upsert
    batch_ids = [
        str(uuid.uuid5(uuid.NAMESPACE_DNS, doc.metadata["chunk_id"]))
        for doc in batch
    ]

    # Inserimento se assetne o upsert implicito se giù esistente il corrispettivo batch_id
    #  NB: se dovessero essere cambiate le dimensioni dei chunk, l'upsert non cancellerà i vecchi punti da Qdrant
    vectorstore.add_documents(documents=batch, ids=batch_ids)

    batch_elapsed = time.perf_counter() - batch_start_time
    processed_count = min(i + batch_size, total_docs)

    print(
        f"   [Batch {batch_idx:02d}] Indicizzati {processed_count:3d}/{total_docs} chunk "
        f"<T> tempo batch: {batch_elapsed:.2f} s"
    )

# Tempo totale
total_elapsed = time.perf_counter() - total_start_time

# Salvataggio associazione collection - embedding_model
#  Ogni collection deve infatti essere manipolata dallo stesso embedding_model
#  che ne ha fatto l'Ingestion
registry_data = {}
if os.path.exists(REGISTRY_PATH):
    try:
        with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
            registry_data = json.load(f)
    except Exception:
        registry_data = {}

registry_data[COLLECTION_NAME] = {
    "embedding_model": EMBEDDING_MODEL,
    "vector_dimension": vector_dim,
    "distance_metric": "COSINE",
    "total_chunks": total_docs,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    #"hypothetical_tags": sorted(list(dataset_global_tags)),
    "updated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

os.makedirs(os.path.dirname(REGISTRY_PATH), exist_ok=True)
with open(REGISTRY_PATH, "w", encoding="utf-8") as f:
    json.dump(registry_data, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 55)
print(
    f"!!!> COMPLETATO! Inseriti {total_docs} chunk nella collezione '{COLLECTION_NAME}'"
)
print(f"<TT> TEMPO TOTALE VETTORIZZAZIONE & UPLOAD: {total_elapsed:.2f} s")
print("=" * 55)

/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]


>> Avvio vettorizzazione di 985 chunk (25 chunk per batch)
   [Batch 01] Indicizzati  25/985 chunk <T> tempo batch: 11.55 s
   [Batch 02] Indicizzati  50/985 chunk <T> tempo batch: 14.20 s
   [Batch 03] Indicizzati  75/985 chunk <T> tempo batch: 12.91 s
   [Batch 04] Indicizzati 100/985 chunk <T> tempo batch: 9.47 s
   [Batch 05] Indicizzati 125/985 chunk <T> tempo batch: 10.38 s
   [Batch 06] Indicizzati 150/985 chunk <T> tempo batch: 10.75 s
   [Batch 07] Indicizzati 175/985 chunk <T> tempo batch: 8.39 s
   [Batch 08] Indicizzati 200/985 chunk <T> tempo batch: 7.78 s
   [Batch 09] Indicizzati 225/985 chunk <T> tempo batch: 10.31 s
   [Batch 10] Indicizzati 250/985 chunk <T> tempo batch: 12.04 s
   [Batch 11] Indicizzati 275/985 chunk <T> tempo batch: 10.06 s
   [Batch 12] Indicizzati 300/985 chunk <T> tempo batch: 11.52 s
   [Batch 13] Indicizzati 325/985 chunk <T> tempo batch: 9.48 s
   [Batch 14] Indicizzati 350/985 chunk <T> tempo batch: 9.09 s
   [Batch 15] Indicizzati 375/985 c